fazer a tratagem das tabelas Olist uma por uma

In [0]:
df = spark.table("olist.bronze.orders")
display(df)
df.printSchema()

In [0]:
columns_data = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for i in columns_data:
    df = df.withColumn(i, df[i].cast("timestamp"))

df = df.dropDuplicates(["order_id"])

df.write.mode("overwrite").saveAsTable("olist.silver.orders")

total = df.count()

print("=== CHECAGEM DE QUALIDADE: olist.silver.orders ===")
print(f"Total de linhas: {total}")

print("\n1) Pedidos duplicados (order_id repetido). Esperado: 0")
duplicados = total - df.select("order_id").distinct().count()
print(f"   Duplicados: {duplicados}")

print("\n2) Nulos por coluna")
for c in df.columns:
    nulos = df.filter(df[c].isNull()).count()
    print(f"   {c}: {nulos} nulos")

print("\n3) Quantidade de pedidos por status")
display(df.groupBy("order_status").count().orderBy("count", ascending=False))

print("\n4) Pedidos SEM data de entrega, por status")
print("   Normal para cancelados e em andamento. Estranho se for 'delivered'.")
display(
    df.filter(df.order_delivered_customer_date.isNull())
      .groupBy("order_status")
      .count()
      .orderBy("count", ascending=False)
)

print("\n5) Datas impossíveis. Esperado: 0 nas duas")
antes_compra = df.filter(df.order_delivered_customer_date < df.order_purchase_timestamp).count()
print(f"   Entregues antes da compra: {antes_compra}")
aprovado_antes = df.filter(df.order_approved_at < df.order_purchase_timestamp).count()
print(f"   Aprovados antes da compra: {aprovado_antes}")

**Observação sobre `silver.orders`:** 8 pedidos com status `delivered` não têm data de entrega ao cliente (`order_delivered_customer_date`). Esses registros foram mantidos na silver, que guarda dados limpos e de uso geral, e serão filtrados apenas nas golds em que a data é necessária, como no modelo de atraso. Os demais pedidos sem data de entrega (2.957) são esperados, pois estão em andamento, cancelados ou indisponíveis.

In [0]:
df_items = spark.table("olist.bronze.order_items")
display(df_items)
df_items.printSchema()

In [0]:
colunas_int = ["order_item_id"]
colunas_double = ["price", "freight_value"]

for c in colunas_int:
    df_items = df_items.withColumn(c, df_items[c].cast("int"))

for c in colunas_double:
    df_items = df_items.withColumn(c, df_items[c].cast("double"))

df_items = df_items.withColumn("shipping_limit_date", df_items["shipping_limit_date"].cast("timestamp"))

df_items = df_items.dropDuplicates(["order_id", "order_item_id"])

df_items.write.mode("overwrite").saveAsTable("olist.silver.order_items")

In [0]:
spark.table("olist.silver.order_items").printSchema()
print(spark.table("olist.silver.order_items").count())


In [0]:
df = spark.table("olist.silver.order_items")
total = df.count()

print("=== CHECAGEM DE QUALIDADE: olist.silver.order_items ===")
print(f"Total de linhas: {total}")

print("\n1) Itens duplicados (order_id + order_item_id). Esperado: 0")
distintos = df.select("order_id", "order_item_id").distinct().count()
print(f"   Duplicados: {total - distintos}")

print("\n2) Nulos por coluna. Esperado: 0 em todas")
print("   Se aparecer nulo em coluna convertida, o cast falhou para essas linhas.")
for c in df.columns:
    print(f"   {c}: {df.filter(df[c].isNull()).count()} nulos")

print("\n3) Valores inválidos de preço e frete")
print(f"   Preço <= 0 (esperado 0): {df.filter(df.price <= 0).count()}")
print(f"   Frete < 0 (esperado 0): {df.filter(df.freight_value < 0).count()}")
print(f"   Frete = 0 (pode ser frete grátis, normal): {df.filter(df.freight_value == 0).count()}")

print("\n4) Itens cujo pedido não existe em silver.orders. Esperado: 0")
orders = spark.table("olist.silver.orders")
print(f"   Itens órfãos: {df.join(orders, 'order_id', 'left_anti').count()}")

print("\n5) Pedidos de silver.orders sem nenhum item")
print("   Pode acontecer em pedidos cancelados ou indisponíveis.")
print(f"   Pedidos sem itens: {orders.join(df, 'order_id', 'left_anti').count()}")

print("\n6) Resumo de preço e frete (mínimo, média, máximo)")
display(df.select("price", "freight_value").summary("min", "mean", "max"))

In [0]:
df_pay = spark.table("olist.bronze.order_payments")
display(df_pay)
df_pay.printSchema()

In [0]:
colunas_int = ["payment_sequential", "payment_installments"]

for c in colunas_int:
    df_pay = df_pay.withColumn(c, df_pay[c].cast("int"))

df_pay = df_pay.withColumn("payment_value", df_pay["payment_value"].cast("double"))

df_pay = df_pay.dropDuplicates(["order_id", "payment_sequential"])

df_pay.write.mode("overwrite").saveAsTable("olist.silver.order_payments")

In [0]:
df = spark.table("olist.silver.order_payments")
total = df.count()

print("=== CHECAGEM DE QUALIDADE: olist.silver.order_payments ===")
print(f"Total de linhas: {total}  (esperado: 103886)")

print("\n1) Pagamentos duplicados (order_id + payment_sequential). Esperado: 0")
distintos = df.select("order_id", "payment_sequential").distinct().count()
print(f"   Duplicados: {total - distintos}")

print("\n2) Nulos por coluna. Esperado: 0 em todas")
print("   Se aparecer nulo em coluna convertida, o cast falhou para essas linhas.")
for c in df.columns:
    print(f"   {c}: {df.filter(df[c].isNull()).count()} nulos")

print("\n3) Tipos de pagamento existentes")
print("   Confira se há algum valor estranho, como 'not_defined'.")
display(df.groupBy("payment_type").count().orderBy("count", ascending=False))

print("\n4) Valores e parcelas suspeitos")
print(f"   Valor < 0 (esperado 0): {df.filter(df.payment_value < 0).count()}")
print(f"   Valor = 0 (investigar): {df.filter(df.payment_value == 0).count()}")
print(f"   Parcelas = 0 (investigar): {df.filter(df.payment_installments == 0).count()}")

print("\n5) Pagamentos de pedidos que não existem em silver.orders. Esperado: 0")
orders = spark.table("olist.silver.orders")
print(f"   Pagamentos órfãos: {df.join(orders, 'order_id', 'left_anti').count()}")

print("\n6) Pedidos de silver.orders sem nenhum pagamento")
print(f"   Pedidos sem pagamento: {orders.join(df, 'order_id', 'left_anti').count()}")

print("\n7) Resumo do valor pago (mínimo, média, máximo)")
display(df.select("payment_value", "payment_installments").summary("min", "mean", "max"))

Observação sobre silver.order_payments: 9 pagamentos têm valor zero, 2 têm parcelas igual a zero e 1 pedido de silver.orders não tem pagamento registrado. Os registros foram mantidos, e cada projeto decide na gold se precisa filtrá-los

In [0]:
df_rev = spark.table("olist.bronze.order_reviews")
display(df_rev)
df_rev.printSchema()

In [0]:
df_rev = df_rev.withColumn("review_score", df_rev["review_score"].cast("int"))

colunas_data = ["review_creation_date", "review_answer_timestamp"]

for c in colunas_data:
    df_rev = df_rev.withColumn(c, df_rev[c].cast("timestamp"))

df_rev.write.mode("overwrite").saveAsTable("olist.silver.order_reviews")

In [0]:
df = spark.table("olist.silver.order_reviews")
total = df.count()

print("=== CHECAGEM DE QUALIDADE: olist.silver.order_reviews ===")
print(f"Total de linhas: {total}  (esperado: 99224)")

print("\n1) Duplicidade")
ids_distintos = df.select("review_id").distinct().count()
pares_distintos = df.select("review_id", "order_id").distinct().count()
print(f"   review_id diferentes: {ids_distintos}")
print(f"   Linhas com review_id repetido: {total - ids_distintos}")
print(f"   Linhas repetidas considerando review_id + order_id (esperado: 0): {total - pares_distintos}")

print("\n2) Nulos por coluna")
print("   Nulo em título e mensagem é normal: o cliente pode dar só a nota.")
print("   Nas demais colunas o esperado é 0.")
for c in df.columns:
    print(f"   {c}: {df.filter(df[c].isNull()).count()} nulos")

print("\n3) Notas (review_score)")
print("   Esperado: só valores de 1 a 5")
display(df.groupBy("review_score").count().orderBy("review_score"))

print("\n4) Datas impossíveis. Esperado: 0")
resposta_antes = df.filter(df.review_answer_timestamp < df.review_creation_date).count()
print(f"   Respondidas antes de a avaliação ser criada: {resposta_antes}")

print("\n5) Avaliações de pedidos que não existem em silver.orders. Esperado: 0")
orders = spark.table("olist.silver.orders")
print(f"   Avaliações órfãs: {df.join(orders, 'order_id', 'left_anti').count()}")

print("\n6) Pedidos de silver.orders sem nenhuma avaliação")
print("   Normal: nem todo cliente avalia.")
print(f"   Pedidos sem avaliação: {orders.join(df, 'order_id', 'left_anti').count()}")

print("\n7) Pedidos com mais de uma avaliação")
por_pedido = df.groupBy("order_id").count().withColumnRenamed("count", "qtd_avaliacoes")
display(
    por_pedido.filter("qtd_avaliacoes > 1")
              .groupBy("qtd_avaliacoes").count()
              .withColumnRenamed("count", "pedidos")
              .orderBy("qtd_avaliacoes")
)

In [0]:
df_prod = spark.table("olist.bronze.products")
display(df_prod)
df_prod.printSchema()

In [0]:
df_trad = spark.table("olist.bronze.category_translation")
display(df_trad)
df_trad.printSchema()

In [0]:
df_prod = (
    df_prod
    .withColumnRenamed("product_name_lenght", "product_name_length")
    .withColumnRenamed("product_description_lenght", "product_description_length")
)

colunas_int = [
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

for c in colunas_int:
    df_prod = df_prod.withColumn(c, df_prod[c].cast("int"))

df_trad = df_trad.dropDuplicates(["product_category_name"])

df_prod = df_prod.join(df_trad, "product_category_name", "left")

df_prod = df_prod.dropDuplicates(["product_id"])

df_prod.write.mode("overwrite").saveAsTable("olist.silver.products")

In [0]:
spark.table("olist.silver.products").printSchema()
print(spark.table("olist.silver.products").count())


In [0]:
df = spark.table("olist.silver.products")
total = df.count()

print("=== CHECAGEM DE QUALIDADE: olist.silver.products ===")
print(f"Total de linhas: {total}  (esperado: 32951)")

print("\n1) Produtos duplicados (product_id repetido). Esperado: 0")
print(f"   Duplicados: {total - df.select('product_id').distinct().count()}")

print("\n2) Nulos por coluna")
print("   Alguns produtos vêm com cadastro incompleto (sem categoria, nome, descrição e fotos).")
for c in df.columns:
    print(f"   {c}: {df.filter(df[c].isNull()).count()} nulos")

print("\n3) Categorias preenchidas mas SEM tradução para o inglês")
display(
    df.filter(df.product_category_name.isNotNull() & df.product_category_name_english.isNull())
      .groupBy("product_category_name").count()
      .orderBy("count", ascending=False)
)

print("\n4) Medidas inválidas (zero ou negativas). Esperado: 0")
for c in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    print(f"   {c} <= 0: {df.filter(df[c] <= 0).count()}")

print("\n5) Itens de silver.order_items cujo produto não existe em silver.products. Esperado: 0")
itens = spark.table("olist.silver.order_items")
print(f"   Itens sem produto: {itens.join(df, 'product_id', 'left_anti').count()}")

print("\n6) Produtos que nunca aparecem em nenhum item")
print(f"   Produtos sem venda: {df.join(itens, 'product_id', 'left_anti').count()}")

print("\n7) Resumo das medidas (mínimo, média, máximo)")
display(
    df.select("product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm")
      .summary("min", "mean", "max")
)


In [0]:
df_geo = spark.table("olist.bronze.geolocation")
display(df_geo)
df_geo.printSchema()

In [0]:
df_geo = (
    df_geo
    .withColumn("geolocation_lat", df_geo["geolocation_lat"].cast("double"))
    .withColumn("geolocation_lng", df_geo["geolocation_lng"].cast("double"))
)

df_geo.write.mode("overwrite").saveAsTable("olist.silver.geolocation")

In [0]:
from pyspark.sql import functions as F

df = spark.table("olist.silver.geolocation")
total = df.count()

print("=== CHECAGEM DE QUALIDADE: olist.silver.geolocation ===")
print(f"Total de linhas: {total}  (esperado: 1000163)")

print("\n1) CEPs (prefixos) diferentes")
print(f"   CEPs distintos: {df.select('geolocation_zip_code_prefix').distinct().count()}")

print("\n2) Nulos por coluna. Esperado: 0 em todas")
for c in df.columns:
    print(f"   {c}: {df.filter(df[c].isNull()).count()} nulos")

print("\n3) Linhas totalmente iguais (todas as colunas repetidas)")
print(f"   Linhas repetidas: {total - df.dropDuplicates().count()}")

print("\n4) CEPs com tamanho diferente de 5 dígitos. Esperado: 0")
print(f"   CEPs fora do padrão: {df.filter('length(geolocation_zip_code_prefix) != 5').count()}")

print("\n5) Pontos fora do Brasil (caixa aproximada, incluindo ilhas). Esperado: poucos")
fora = df.filter(
    ~df.geolocation_lat.between(-34, 6) | ~df.geolocation_lng.between(-74, -28)
).count()
print(f"   Pontos fora do Brasil: {fora}")

print("\n6) Quantos pontos existem por CEP (mínimo, média, máximo)")
display(
    df.groupBy("geolocation_zip_code_prefix").count()
      .select("count").summary("min", "mean", "max")
)

print("\n7) Estados. Esperado: 27")
print(f"   Estados distintos: {df.select('geolocation_state').distinct().count()}")

print("\n8) CEPs com mais de uma grafia de cidade")
print("   Ex.: 'sao paulo' e 'são paulo' para o mesmo CEP.")
multi = (df.groupBy("geolocation_zip_code_prefix")
           .agg(F.countDistinct("geolocation_city").alias("n"))
           .filter("n > 1").count())
print(f"   CEPs com mais de uma grafia: {multi}")



In [0]:
%sql
SHOW TABLES IN olist.silver;

In [0]:
df_customers = spark.table("olist.bronze.customers")
display(df_customers)
df_customers.printSchema()

In [0]:
df_customers = df_customers.dropDuplicates(["customer_id"])
df_customers.write.mode("overwrite").saveAsTable("olist.silver.customers")
print(df_customers.count())

In [0]:
d = spark.table("olist.silver.customers")
total = d.count()

print("=== CHECAGEM DE QUALIDADE: olist.silver.customers ===")
print(f"Total de linhas: {total}  (esperado: 99441)")

print("\n1) customer_id repetido. Esperado: 0")
print(f"   Repetidos: {total - d.select('customer_id').distinct().count()}")

print("\n2) Nulos por coluna. Esperado: 0 em todas")
for c in d.columns:
    print(f"   {c}: {d.filter(d[c].isNull()).count()} nulos")

print("\n3) Clientes sem pedido em silver.orders. Esperado: 0")
orders = spark.table("olist.silver.orders")
print(f"   Clientes sem pedido: {d.join(orders, 'customer_id', 'left_anti').count()}")

print("\n4) Pessoas diferentes (customer_unique_id)")
print("   O customer_id muda a cada pedido, e o customer_unique_id identifica a pessoa.")
print(f"   Pessoas únicas: {d.select('customer_unique_id').distinct().count()}")

In [0]:
df_sellers = spark.table("olist.bronze.sellers")
display(df_sellers)
df_sellers.printSchema()

In [0]:
df_sellers = df_sellers.dropDuplicates(["seller_id"])
df_sellers.write.mode("overwrite").saveAsTable("olist.silver.sellers")

In [0]:
from pyspark.sql import functions as F

d = spark.table("olist.silver.sellers")
total = d.count()

print("=== CHECAGEM DE QUALIDADE: olist.silver.sellers ===")
print(f"Total de linhas: {total}  (esperado: 3095)")

print("\n1) seller_id repetido. Esperado: 0")
print(f"   Repetidos: {total - d.select('seller_id').distinct().count()}")

print("\n2) Nulos por coluna. Esperado: 0 em todas")
for c in d.columns:
    print(f"   {c}: {d.filter(d[c].isNull()).count()} nulos")

print("\n3) CEPs com tamanho diferente de 5 dígitos. Esperado: 0")
print(f"   CEPs fora do padrão: {d.filter('length(seller_zip_code_prefix) != 5').count()}")

print("\n4) Estados")
print("   Esperado: menos de 27, pois nem todo estado tem vendedor.")
print(f"   Estados distintos: {d.select('seller_state').distinct().count()}")
display(d.groupBy("seller_state").count().orderBy("count", ascending=False))

print("\n5) Vendedores de silver.order_items que não existem em silver.sellers. Esperado: 0")
itens = spark.table("olist.silver.order_items")
print(f"   Itens sem vendedor: {itens.join(d, 'seller_id', 'left_anti').count()}")

print("\n6) Vendedores que nunca aparecem em nenhum item")
print(f"   Vendedores sem venda: {d.join(itens, 'seller_id', 'left_anti').count()}")

print("\n7) Cidades com aparência suspeita (têm números, barra ou arroba)")
print("   O campo de cidade é digitado à mão e costuma ter sujeira.")
display(
    d.filter(d.seller_city.rlike("[0-9/@\\\\]"))
     .select("seller_id", "seller_city", "seller_state")
)

print("\n8) CEPs de vendedores que não existem em silver.geolocation")
geo_zips = (spark.table("olist.silver.geolocation")
              .select(F.col("geolocation_zip_code_prefix").alias("zip")).distinct())
cep_vend = d.select(F.col("seller_zip_code_prefix").alias("zip")).distinct()
print(f"   CEPs de vendedores sem geolocation: {cep_vend.join(geo_zips, 'zip', 'left_anti').count()}")

In [0]:
%sql
SHOW TABLES IN olist.silver;